<a href="https://colab.research.google.com/github/osoria80/05MIAR-Aprendizaje-supervisado/blob/main/Script_C1_Oscar_Soria_Corral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EXAMEN - Convocatoria 1 - Programación
Resolver el siguientes problemas de regresión lineal y clasificación.

## 1. Problema de Regresión lineal:

### 1.1. Generación de los datos:

Mediante la función `make_regressión` de sklearn, consultar su como emplearla para generar un conjunto de 2000 datos, 10 características, un ruido de 1 y fijar el parámetro ramdom_state=42.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression

x, y = make_regression(n_samples=2000, n_features=10, n_informative=10, noise=1.0, random_state=42)


### 1.2. Partición de datos externa.
Realizar una partición externa de tipo hold-out seleccionando un 20% de los datos para test (fijar una semilla en 42).

In [2]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print(f"Tamaño train: {x_train.shape}")
print(f"Tamaño test: {x_test.shape}")

Tamaño train: (1600, 10)
Tamaño test: (400, 10)


### 1.3. Estandarización de los datos de train y test.
Utilizar el método StandardScaler().

In [3]:
from sklearn import preprocessing
std_scaling = preprocessing.StandardScaler()

x_train = std_scaling.fit_transform(x_train)
x_test = std_scaling.transform(x_test)

print(f'Max X_train {x_train.max()}, Min X_train {x_train.min()}')

Max X_train 4.518542651424377, Min X_train -3.8165321748795944


### 1.4. Creación de modelos e hiperparametrización mediante el método de sklearn GridSearchCV:

Instrucciones:

- Cree los siguientes modelos de regresión en un diccionario:
  - Regresión Lineal (OLS).
  - Regresión Lasso
  - Regresión Rígida
  - Vecinos más cercanos (KNN)

- Cree las siguiente métricas en un diccionario para evaluar:
  - MAE
  - MSE
  - RMSE
  - R2

- Haga una búsqueda de los siguientes hiperparámetros mediante GridSearchCV:

```
# Hiperparametros
parameters = {'OLS':{},
              'Lasso':{'alpha':(0.1,0.5,1,5,10,50,100)},
              'Ridge':{'alpha':(0.1,0.5,1,5,10,50,100)},
              'KNN':{'n_neighbors':np.arange(1,15),
                     'weights':('uniform','distance'),
                     'metric':('euclidean','manhattan','cosine')
                     }
              }
```

In [4]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import make_scorer, mean_squared_error, mean_absolute_error, r2_score

from math import sqrt

# Modelos
models = {
    'OLS': LinearRegression(),
    'Lasso': Lasso(),
    'Ridge': Ridge(),
    'KNN': KNeighborsRegressor()
}

# Hiperparametros
parameters = {'OLS':{},
              'Lasso':{'alpha':(0.1,0.5,1,5,10,50,100)},
              'Ridge':{'alpha':(0.1,0.5,1,5,10,50,100)},
              'KNN':{'n_neighbors':np.arange(1,15),
                     'weights':('uniform','distance'),
                     'metric':('euclidean','manhattan','cosine')
                     }
              }
# Métricas
metricas = {
  'MAE': 'neg_mean_absolute_error',
  'MSE': 'neg_mean_squared_error',
  'R2' : 'r2',
  'RMSE': make_scorer(lambda y, y_pred:
                      sqrt(mean_squared_error(y, y_pred)),
                      greater_is_better=False),
            }

grid_results = {}

for nombre, modelo in models.items():

    grid = GridSearchCV(
        estimator=modelo,
        param_grid=parameters[nombre],
        scoring=metricas,
        refit='RMSE'
    )

    grid.fit(x_train, y_train)

    grid_results[nombre] = grid

    print(f"{nombre} mejores parámetros: {grid.best_params_}")

OLS mejores parámetros: {}
Lasso mejores parámetros: {'alpha': 0.1}
Ridge mejores parámetros: {'alpha': 0.1}
KNN mejores parámetros: {'metric': 'cosine', 'n_neighbors': np.int64(14), 'weights': 'distance'}


### 1.5. Evaluación de los modelos sobre el conjunto de test.
- Entrenar los modelos anteriores utilizando todos los datos de entrenamiento y quédese con el mejor modelo de cada uno de los modelos de regresión arrojados por GridSearchCV. Es decir, debe tener un mejor modelo por cada regresión (OLS, Lasso, Rígida y KNN)
- Evaluar su rendimiento sobre el conjunto de test mostrando para cada uno de los modelos

In [5]:
resultados = []

for nombre, grid in grid_results.items():

    best_model = grid.best_estimator_

    y_pred = best_model.predict(x_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    resultados.append({
        'Modelo': nombre,
        'Mejores parámetros': grid.best_params_,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2': r2
    })

resultados_df = pd.DataFrame(resultados)

display(resultados_df)

,Modelo,Mejores parámetros,MAE,MSE,RMSE,R2
0,OLS,{},0.773148,0.953421,0.976433,0.999973
1,Lasso,{'alpha': 0.1},0.803091,1.034922,1.017311,0.999971
2,Ridge,{'alpha': 0.1},0.772655,0.952406,0.975913,0.999973
3,KNN,"{'metric': 'cosine', 'n_neighbors': 14, 'weigh...",50.040133,4579.891556,67.674896,0.872031


### 1.6. Interpretación de resultados.
* Justifica cuál de los modelos utilizarías para ponerlo en producción

Para seleccionar el mejor modelo, utilizamos las métricas obtenidas en el conjunto de test. Especialmente RMSE y MSE, ya que penalizan los errores más grandes.

Por lo tanto, buscamos:
* Un RMSE / MSE bajos, nos indican un menor error en las predicciones.
* Un $R^2$ alto, nos indica que le modelo tiene una buena capaz de explicar el comportamiento de la variable objetivo a partir de las variables de entrada.

De entre los modelos, elegiría la regresión Ridge por los siguientes motivos:

* Presenta el menor RMSE (0.9759) y menor MSE (0.9524) de entre todos los modelos.
* Su MAE (0.7727) también es el más bajo.
* Su $R^2$ (0.999973) indica una capacidad explicativa práctocamente total.

## 2. Problema de Clasificación con generación de datos sintéticos y selección de características

En esta actividad se trabajará con generación de datos sintéticos para problemas de clasificación utilizando la función `make_classification` de la librería **scikit-learn**.

El objetivo es evaluar si distintos métodos de selección de características y modelos de clasificación son capaces de identificar las variables realmente relevantes dentro de un conjunto de datos generado artificialmente.

### 2.1 Importación de librerías

Importe las siguientes librerías:

In [6]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif

from sklearn.preprocessing import StandardScaler

### 2.2. Generación del conjunto de datos

2.2.1. Utilice `make_classification` para generar un dataset de clasificación binaria.

Debe definir explícitamente los siguientes parámetros:

- `n_samples`
- `n_features`
- `n_informative`
- `n_redundant`

El dataset debe tener:

- al menos **500 observaciones**
- al menos **10 características**

El objetivo es que existan variables:

- **informativas**
- **redundantes**
- **no informativas**

2.2.2. Explique brevemente los valores elegidos para cada parámetro.


In [8]:
X, y = make_classification(
    n_samples=1000,
    n_features=12,
    n_informative=5,
    n_redundant=3,
    n_repeated=0,
    n_classes=2,
    random_state=42
)


**n_samples = 1000:** Supera el mínimo establecido y proporciona suficientes datos para entrenar y evaluar los modelos con fiabilidad.

**n_features = 12:** Supera el mínimo establecido y permite distribuir los tres tipos de variables (informativas, redundantes y no informativas) con margen.

**n_informative = 5:** Fija que cantidad cercana a la mitad de las características sean realmente
útiles para predecir la clase, lo que hace el problema resoluble pero no aleatorio.

**n_redundant = 3:** Se introduce multicolinealidad en el dataset, ya que estas características
son combinaciones lineales de las informativas. Esto dificulta la selección de variables y
permite evaluar si los métodos son capaces de detectarlas.

Las **4 características restantes** (12 − 5 − 3) son no informativas, es decir,
valores sin relación con la clase.

Por último, fijamos **random_state=42** para garantizar que el dataset
generado sea reproducible en cualquier ejecución.

### 2.3. División de los datos.

Divida el dataset en:

- **conjunto de entrenamiento**
- **conjunto de prueba**

El conjunto de entrenamiento se utilizará para:

- selección de características
- búsqueda de hiperparámetros
- entrenamiento de los modelos

El conjunto de prueba se utilizará **únicamente para evaluar el desempeño final de los modelos**.

In [11]:
# Separamos los datos en dos conjuntos: 20% test y 80% train
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### 2.4. Selección de caractarística

Aplique un método de selección de características para identificar cuáles variables parecen ser más relevantes para la predicción.

Puede utilizar el método **SelectKBest** de `sklearn.feature_selection`. Este método evalúa cada característica de forma individual mediante una medida estadística y selecciona las **k variables con mayor puntuación**, es decir, aquellas que tienen mayor relación con la variable objetivo.

Utilice como función de puntuación `f_classif`.

Compare las características seleccionadas con las definidas como **informativas** en `make_classification` y discuta si el método logra identificarlas correctamente.

In [13]:
selector = SelectKBest(score_func=f_classif, k=5)
selector.fit(X_train, y_train)

# Guardamos los índices de las 5 características seleccionadas
caracteristicas_seleccionadas = np.where(selector.get_support())[0]

print("Características seleccionadas por SelectKBest:", caracteristicas_seleccionadas)

Características seleccionadas por SelectKBest: [ 1  4  6  8 10]


Las características seleccionadas por **SelectKBest** son [1, 4, 6, 8, 10], mientras que las
variables realmente informativas definidas en make_classification son [0, 1, 2, 3, 4].

El método solo identifica correctamente **2 de las 5 variables informativas** (la 1 y la 4),
lo que supone un resultado pobre. Las variables informativas 0, 2 y 3 no fueron seleccionadas.

En su lugar, el método seleccionó la característica 6 (redundante) y las características 8 y 10
(no informativas), lo que indica que el método asignó puntuaciones F altas a variables que
en realidad no aportan información independiente sobre la clase.

## 2.5. Escalado de datos

Antes de entrenar los modelos, estandarice las variables utilizando **StandardScaler** de `sklearn.preprocessing`.

La estandarización consiste en transformar las variables para que tengan:

- media igual a 0
- desviación estándar igual a 1

Este paso es importante para modelos sensibles a la escala de los datos, como **regresión logística** y **SVM**. Los árboles de decisiones, son independientes del escalado.

El escalado debe realizarse de la siguiente manera:

1. Ajustar el `StandardScaler` **únicamente con el conjunto de entrenamiento**.
2. Aplicar esa transformación tanto al conjunto de entrenamiento como al conjunto de prueba.


In [15]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

### 2.6. Modelos e hiperparametrización

Trabaje con los siguientes modelos.

**Modelos obligatorios**

- Árbol de decisión (`DecisionTreeClassifier`)
- Regresión logística (`LogisticRegression`)

**Modelo bonus**

- Máquina de soporte vectorial (`SVC`)

Para cada modelo, realice una búsqueda de hiperparámetros utilizando **GridSearchCV** sobre el **conjunto de entrenamiento**.

Utilice las siguientes listas de hiperparámetros:

**Árbol de decisión**

- `max_depth`: [3, 5, 10, None]
- `min_samples_split`: [2, 5, 10]
- `criterion`: ["gini", "entropy"]

**Regresión logística**

- `C`: [0.01, 0.1, 1, 10]
- `solver`: ["lbfgs", "liblinear"]

**SVM (bonus)**

- `C`: [0.1, 1, 10]
- `kernel`: ["linear", "rbf"]
- `gamma`: ["scale", "auto"]

Utilice validación cruzada dentro de `GridSearchCV` para determinar la mejor combinación de hiperparámetros.


In [17]:
# Árbol de decisión
params_arbol = {
    "max_depth"        : [3, 5, 10, None],
    "min_samples_split": [2, 5, 10],
    "criterion"        : ["gini", "entropy"]
}

grid_arbol = GridSearchCV(DecisionTreeClassifier(random_state=42), params_arbol, cv=5, scoring="accuracy")
grid_arbol.fit(X_train, y_train)

# Regresión logística
params_rl = {
    "C"     : [0.01, 0.1, 1, 10],
    "solver": ["lbfgs", "liblinear"]
}

grid_rl = GridSearchCV(LogisticRegression(random_state=42, max_iter=1000), params_rl, cv=5, scoring="accuracy")
grid_rl.fit(X_train_scaled, y_train)

# Resultados
print("Árbol de decisión")
print("  Mejores hiperparámetros:", grid_arbol.best_params_)
print()
print("Regresión logística")
print("  Mejores hiperparámetros:", grid_rl.best_params_)

Árbol de decisión
  Mejores hiperparámetros: {'criterion': 'entropy', 'max_depth': 5, 'min_samples_split': 2}

Regresión logística
  Mejores hiperparámetros: {'C': 0.1, 'solver': 'liblinear'}


### 2.7. Evaluación y análisis

Para cada modelo:

- Del punto anterior, obtenga el modelo con **los mejores hiperparámetros**.
- Evalúe el modelo final en el **conjunto de prueba**.
- Realice este proceso en dos escenarios:
  - utilizando **todas las características**
  - utilizando **solo las características seleccionadas**
- Compare los resultados utilizando métricas de clasificación apropiada y discuta:
  - si la selección de características mejora el desempeño del modelo
  - si reduce la complejidad del modelo
  - si permite identificar correctamente las variables informativas generadas en el dataset
  - qué modelo obtiene mejor desempeño después del ajuste de hiperparámetros


In [18]:
# Seleccionamos los datos con características seleccionadas
X_train_sel        = X_train[:, caracteristicas_seleccionadas]
X_test_sel         = X_test[:, caracteristicas_seleccionadas]
X_train_scaled_sel = X_train_scaled[:, caracteristicas_seleccionadas]
X_test_scaled_sel  = X_test_scaled[:, caracteristicas_seleccionadas]

# Modelos con mejores hiperparámetros
mejor_arbol = DecisionTreeClassifier(
    criterion         = grid_arbol.best_params_["criterion"],
    max_depth         = grid_arbol.best_params_["max_depth"],
    min_samples_split = grid_arbol.best_params_["min_samples_split"],
    random_state      = 42
)

mejor_rl = LogisticRegression(
    C            = grid_rl.best_params_["C"],
    solver       = grid_rl.best_params_["solver"],
    random_state = 42,
    max_iter     = 1000
)

# Árbol de decisión
mejor_arbol.fit(X_train, y_train)
y_pred_arbol_todas = mejor_arbol.predict(X_test)

mejor_arbol.fit(X_train_sel, y_train)
y_pred_arbol_sel = mejor_arbol.predict(X_test_sel)

# Regresión logística
mejor_rl.fit(X_train_scaled, y_train)
y_pred_rl_todas = mejor_rl.predict(X_test_scaled)

mejor_rl.fit(X_train_scaled_sel, y_train)
y_pred_rl_sel = mejor_rl.predict(X_test_scaled_sel)

# Visualizamos los resultados
print("ÁRBOL DE DECISIÓN - Todas las características")
print(classification_report(y_test, y_pred_arbol_todas))

print("ÁRBOL DE DECISIÓN - Características seleccionadas")
print(classification_report(y_test, y_pred_arbol_sel))

print("REGRESIÓN LOGÍSTICA - Todas las características")
print(classification_report(y_test, y_pred_rl_todas))

print("REGRESIÓN LOGÍSTICA - Características seleccionadas")
print(classification_report(y_test, y_pred_rl_sel))

ÁRBOL DE DECISIÓN - Todas las características
              precision    recall  f1-score   support

           0       0.85      0.92      0.89       101
           1       0.91      0.84      0.87        99

    accuracy                           0.88       200
   macro avg       0.88      0.88      0.88       200
weighted avg       0.88      0.88      0.88       200

ÁRBOL DE DECISIÓN - Características seleccionadas
              precision    recall  f1-score   support

           0       0.85      0.89      0.87       101
           1       0.88      0.84      0.86        99

    accuracy                           0.86       200
   macro avg       0.87      0.86      0.86       200
weighted avg       0.87      0.86      0.86       200

REGRESIÓN LOGÍSTICA - Todas las características
              precision    recall  f1-score   support

           0       0.83      0.79      0.81       101
           1       0.80      0.84      0.82        99

    accuracy                          

### Análisis

**Mejora del desempeño del modelo:**
En ambos modelos, usar las características seleccionadas reduce ligeramente el desempeño.
El árbol de decisión pasa de 0.88 a 0.86 de accuracy, y la regresión logística de 0.81 a 0.80.
La selección de características no mejora los modelos en este caso, lo cual es consistente
con los resultados del apartado 2.4: **SelectKBest** solo identificó correctamente 2 de las 5
variables informativas, por lo que el subconjunto seleccionado descarta información relevante.


**Complejidad del modelo:**
La comlejidad del modelo se reduce notablemente. Ambos modelos trabajan con 5 características en lugar de 12, lo que reduce la complejidad. El árbol genera estructuras más pequeñas y la regresión
logística utiliza menos coeficientes.


**Identificación correcta de las variables informativas:**
No se identifican correctamente. Como se observó en el apartado 2.4, **SelectKBest** seleccionó las características
[1, 4, 6, 8, 10], de las cuales solo 1 y 4 son realmente informativas. Las características
6, 8 y 10 se incluyeron sin ser informativas, mientras que las
informativas 0, 2 y 3 quedaron descartadas. Esto explica la caída de rendimiento al usar
las características seleccionadas.


**Modelo con mejor desempeño:**
El árbol de decisión obtiene mejor desempeño en ambos escenarios (0.88 frente a 0.81 con
todas las características). Esto puede resultar sorprendente dado que make_classification
genera datos con relaciones lineales, que en principio favorecen a la regresión logística.
Sin embargo, el ajuste de hiperparámetros mediante GridSearchCV permite al árbol encontrar una configuración adecuada para el dataset generado.